<a href="https://colab.research.google.com/github/obaidah3/rag-ecommerce-chatbot/blob/main/05_deploy_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5. Deployment on Google Colab


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/chatbot_project"
MODELS_DIR = f"{PROJECT_DIR}/models"
print("Loading models from:", MODELS_DIR)

Mounted at /content/drive
Loading models from: /content/drive/MyDrive/chatbot_project/models


In [2]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio sentence-transformers faiss-cpu groq transformers joblib pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.4 MB/s eta 0:00:00


In [3]:
import os
os.environ["GROQ_API_KEY"] = "****************************************************"
NGROK_AUTHTOKEN = "*********************************************************"


In [4]:
# Load all 4 models (same logic as app.py, paths point at Drive)
import joblib, faiss, pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import pipeline as hf_pipeline
from groq import Groq

lang_model = joblib.load(f"{MODELS_DIR}/language_detector.joblib")
intent_model = joblib.load(f"{MODELS_DIR}/intent_classifier.joblib")
sentiment_pipe = hf_pipeline("text-classification", model=f"{MODELS_DIR}/sentiment_model", tokenizer=f"{MODELS_DIR}/sentiment_model")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
faiss_index = faiss.read_index(f"{MODELS_DIR}/rag_index.faiss")
corpus_df = pd.read_parquet(f"{MODELS_DIR}/rag_corpus.parquet")
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("all models loaded")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

all models loaded


In [5]:
SYSTEM_PROMPT = '''You are a helpful, professional customer support assistant for an online retailer.
Answer the customer's question using ONLY the information in the retrieved support responses below.
If the customer sounds frustrated ({sentiment}), acknowledge that before answering.
If the retrieved context does not cover the question, say so honestly and offer to escalate to a human agent rather than guessing.'''

def retrieve(query, k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = faiss_index.search(q_emb, k)
    return [{"instruction": corpus_df.iloc[i]["instruction"], "response": corpus_df.iloc[i]["response"], "score": float(s)}
            for s, i in zip(scores[0], idxs[0])]

def rag_answer(user_message, sentiment):
    chunks = retrieve(user_message, k=3)
    context = "\n\n".join(f"- Q: {c['instruction']}\n  A: {c['response']}" for c in chunks)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(sentiment=sentiment)},
        {"role": "user", "content": f'Context (retrieved past support responses):\n{context}\n\nCustomer question: "{user_message}"'}
    ]
    resp = groq_client.chat.completions.create(model="openai/gpt-oss-20b", messages=messages, temperature=0.3)
    return {"answer": resp.choices[0].message.content, "sources": chunks}


In [6]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional, List

app = FastAPI(title="E-commerce Support Chatbot")

SMALL_TALK_REPLY = "You're welcome! Is there anything else I can help you with today?"

class ChatRequest(BaseModel):
    message: str

class ChatResponse(BaseModel):
    language: str
    sentiment: str
    intent: str
    reply: str
    escalate: bool
    sources: Optional[List[dict]] = None

@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    text = req.message
    language = lang_model.predict([text])[0]
    sentiment = sentiment_pipe(text[:512])[0]["label"]
    intent = intent_model.predict([text])[0]

    escalate = False
    sources = None

    if intent == "small_talk":
        reply = SMALL_TALK_REPLY
    elif intent == "complaint":
        rag_result = rag_answer(text, sentiment)
        reply = "I'm really sorry to hear about this experience — that's not the standard we want for you. " + rag_result["answer"] + "\n\nI'm also flagging this to a human agent so we can follow up personally."
        escalate = True
        sources = rag_result["sources"]
    elif intent == "out_of_scope":
        reply = "I'm not able to help with that directly through this assistant, but I can connect you with a human agent who can. Would you like me to do that?"
        escalate = True
    else:
        rag_result = rag_answer(text, sentiment)
        reply = rag_result["answer"]
        sources = rag_result["sources"]

    return ChatResponse(language=language, sentiment=sentiment, intent=intent, reply=reply, escalate=escalate, sources=sources)

@app.get("/health")
def health():
    return {"status": "ok"}


In [ ]:
import nest_asyncio
nest_asyncio.apply()

import uvicorn
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_AUTHTOKEN
nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

Public URL: NgrokTunnel: "https://unproven-overdrive-hurry.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [3561]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     197.135.134.122:0 - "POST /chat HTTP/1.1" 422 Unprocessable Content
INFO:     197.135.134.122:0 - "POST /chat HTTP/1.1" 200 OK




```
curl -X POST <PUBLIC_URL>/chat -H "Content-Type: application/json" \
  -d '{"message": "Where is my order #4521?"}'
```
